<a href="https://colab.research.google.com/github/kumarmaheshbly-oss/Project-Aura-/blob/main/aura_shield_v21_maximum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:

# 1. Install Libraries
!pip install "orekit-jpype[jdk4py]" -q

import os
import jdk4py
import jpype
import orekit_jpype as orekit

# 2. Set Java Environment
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)

# 3. JVM Start Karein (Sahi Tarika: JVM pehle, phir Java imports)
if not jpype.isJVMStarted():
    print("Starting JVM...")
    orekit.initVM()
    print("✅ JVM Started!")

# 4. Ab pyhelpers import karein (JVM start hone ke baad)
from orekit_jpype.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir

# 5. Orekit Data Download & Setup
print("Downloading Orekit Data (Wait 1-2 mins)...")
download_orekit_data_curdir()
setup_orekit_curdir()

print("✅ Orekit Engine aur IERS Data Ready! Ab aap apna v21 code run kar sakte hain.")

✅ Orekit Engine aur IERS Data Ready! Ab aap apna v21 code run kar sakte hain.


In [12]:

# --- AURA-SHIELD v21.0 (Ready for Colab) ---
from org.orekit.propagation.analytical.tle import TLE, TLEPropagator
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.frames import FramesFactory

import numpy as np
import urllib.request
import ssl
import json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================
class Config:
    DURATION_HOURS     = 24
    COARSE_STEP_MIN    = 2
    FINE_WINDOW_MIN    = 10
    FINE_STEP_SEC      = 4
    HBR                = 0.035
    MC_SAMPLES         = 80000
    DEFAULT_SIGMA      = np.array([0.28, 0.95, 0.38])
    MAX_SECONDARIES    = 12
    MAX_WORKERS        = 6
    REPORT_FILE        = "AURA_SHIELD_v21_Maximum_Report.json"

cfg = Config()

# ============================================================
# DATA LAYER
# ============================================================
def fetch(url: str):
    try:
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, context=ctx, timeout=15) as r:
            return r.read().decode()
    except Exception as e:
        print(f"[Data Error] {e}")
        return None

def load_iss_tle():
    raw = fetch("https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=tle")
    if raw:
        lines = raw.strip().splitlines()
        if len(lines) >= 3:
            print(f"[TLE] Primary loaded → {lines[0].strip()}")
            return lines[1].strip(), lines[2].strip()
    print("[TLE] Fallback ISS TLE used")
    return ("1 25544U 98067A   26304.17644455  .00006183  00000+0  11976-3 0  9995",
            "2 25544  51.6332 347.1486 0007016 330.4977  29.5614 15.49124187592838")

def load_starlink_tles(max_count=12):
    raw = fetch("https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle")
    tles = {}
    if raw:
        lines = raw.strip().splitlines()
        i = 0
        while i < len(lines) - 2 and len(tles) < max_count:
            name = lines[i].strip()
            if "STARLINK" in name.upper():
                tles[name] = (lines[i+1].strip(), lines[i+2].strip())
            i += 3
    if not tles:
        tles["STARLINK-1008"] = (
            "1 44714U 19074B   26303.25834137  .00114493  00000+0  98569-3 0  9999",
            "2 44714  53.1439 131.8443 0001732 163.1296 196.9779 15.68475578385305"
        )
    print(f"[TLE] Secondary objects loaded → {len(tles)}")
    return tles

# ============================================================
# OREKIT CORE
# ============================================================
def create_propagator(line1, line2):
    return TLEPropagator.selectExtrapolator(TLE(line1, line2))

def get_pv(propagator, date):
    pv = propagator.getPVCoordinates(date, FramesFactory.getEME2000())
    pos = np.array([pv.getPosition().getX(),
                    pv.getPosition().getY(),
                    pv.getPosition().getZ()]) / 1000.0
    vel = np.array([pv.getVelocity().getX(),
                    pv.getVelocity().getY(),
                    pv.getVelocity().getZ()]) / 1000.0
    return pos, vel

# ============================================================
# ANALYSIS ENGINE
# ============================================================
def compute_rtn(rel_pos, rel_vel):
    r_hat = rel_pos / (np.linalg.norm(rel_pos) + 1e-12)
    v_hat = rel_vel / (np.linalg.norm(rel_vel) + 1e-12)
    c_hat = np.cross(r_hat, v_hat)
    c_hat /= (np.linalg.norm(c_hat) + 1e-12)
    i_hat = np.cross(c_hat, r_hat)
    return (float(np.dot(rel_pos, r_hat)),
            float(np.dot(rel_pos, i_hat)),
            float(np.dot(rel_pos, c_hat)))

def monte_carlo_pc(rel_pos, sigma):
    samples = np.random.normal(0, 1, (cfg.MC_SAMPLES, 3)) * sigma
    distances = np.linalg.norm(samples + rel_pos, axis=1)
    return float(np.sum(distances < cfg.HBR) / cfg.MC_SAMPLES)

def analyze_one(primary, sec_tle, name, start):
    prop = create_propagator(*sec_tle)

    best_dist = 1e9
    best_date = start
    for m in range(0, cfg.DURATION_HOURS * 60, cfg.COARSE_STEP_MIN):
        date = start.shiftedBy(m * 60.0)
        p1, _ = get_pv(primary, date)
        p2, _ = get_pv(prop, date)
        d = np.linalg.norm(p1 - p2)
        if d < best_dist:
            best_dist = d
            best_date = date

    min_dist = 1e9
    tca = best_date
    rel_pos = rel_vel = None

    for s in range(-cfg.FINE_WINDOW_MIN * 60, cfg.FINE_WINDOW_MIN * 60 + 1, cfg.FINE_STEP_SEC):
        date = best_date.shiftedBy(float(s))
        p1, v1 = get_pv(primary, date)
        p2, v2 = get_pv(prop, date)
        d = np.linalg.norm(p1 - p2)
        if d < min_dist:
            min_dist = d
            tca = date
            rel_pos = p1 - p2
            rel_vel = v1 - v2

    radial, in_track, cross = compute_rtn(rel_pos, rel_vel)
    pc = monte_carlo_pc(rel_pos, cfg.DEFAULT_SIGMA)

    if pc >= 1e-4: risk = "RED"
    elif pc >= 1e-5: risk = "ORANGE"
    elif pc >= 1e-6: risk = "YELLOW"
    else: risk = "GREEN"

    return {
        "secondary": name,
        "tca": str(tca),
        "miss_distance_m": round(min_dist * 1000, 1),
        "relative_speed_km_s": round(float(np.linalg.norm(rel_vel)), 4),
        "rtn_m": {
            "radial": round(radial * 1000, 1),
            "in_track": round(in_track * 1000, 1),
            "cross_track": round(cross * 1000, 1)
        },
        "pc": pc,
        "risk": risk
    }

# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 78)
    print(" AURA-SHIELD v21.0  |  MAXIMUM COMPLETE EDITION")
    print("=" * 78)

    utc = TimeScalesFactory.getUTC()
    start = AbsoluteDate(2026, 10, 31, 12, 0, 0.0, utc)

    iss_l1, iss_l2 = load_iss_tle()
    primary = create_propagator(iss_l1, iss_l2)
    secondaries = load_starlink_tles(cfg.MAX_SECONDARIES)

    print("\n[Engine] Parallel screening started...")
    results = []

    with ThreadPoolExecutor(max_workers=cfg.MAX_WORKERS) as executor:
        futures = {
            executor.submit(analyze_one, primary, tle, name, start): name
            for name, tle in secondaries.items()
        }
        for fut in as_completed(futures):
            res = fut.result()
            results.append(res)
            print(f"  {res['secondary']:<34} Pc={res['pc']:.2e}  [{res['risk']}]")

    results.sort(key=lambda x: x["pc"], reverse=True)

    summary = {"RED": 0, "ORANGE": 0, "YELLOW": 0, "GREEN": 0}
    for r in results:
        summary[r["risk"]] += 1

    report = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "version": "AURA-SHIELD v21.0 Maximum",
        "primary": "ISS (ZARYA)",
        "total_objects_screened": len(results),
        "risk_summary": summary,
        "results": results
    }

    with open(cfg.REPORT_FILE, "w") as f:
        json.dump(report, f, indent=2)

    print("\n" + "=" * 78)
    print(" RISK SUMMARY")
    print("=" * 78)
    for k, v in summary.items():
        print(f"  {k:<8}: {v}")

    print("\nTOP CONJUNCTIONS")
    print("-" * 78)
    for r in results[:7]:
        print(f"{r['secondary']:<34} | {r['miss_distance_m']:>7.1f} m | {r['pc']:.2e} | {r['risk']}")

    print("\n" + "=" * 78)
    print(f"Full report saved → {cfg.REPORT_FILE}")
    print("=" * 78)

# Run the simulation
main()

 AURA-SHIELD v21.0  |  MAXIMUM COMPLETE EDITION
[TLE] Primary loaded → ISS (ZARYA)
[TLE] Secondary objects loaded → 12

[Engine] Parallel screening started...
  STARLINK-1012                      Pc=0.00e+00  [GREEN]
  STARLINK-1017                      Pc=0.00e+00  [GREEN]
  STARLINK-1042                      Pc=0.00e+00  [GREEN]
  STARLINK-1020                      Pc=0.00e+00  [GREEN]
  STARLINK-1039                      Pc=0.00e+00  [GREEN]
  STARLINK-1008                      Pc=0.00e+00  [GREEN]
  STARLINK-1043                      Pc=0.00e+00  [GREEN]
  STARLINK-1048                      Pc=0.00e+00  [GREEN]
  STARLINK-1047                      Pc=0.00e+00  [GREEN]
  STARLINK-1063                      Pc=0.00e+00  [GREEN]
  STARLINK-1114                      Pc=0.00e+00  [GREEN]
  STARLINK-1046                      Pc=0.00e+00  [GREEN]

 RISK SUMMARY
  RED     : 0
  ORANGE  : 0
  YELLOW  : 0
  GREEN   : 12

TOP CONJUNCTIONS
-------------------------------------------------------